# Gemma-3 4B Fine-tune (Colab)Unsloth • RAG MDN

In [4]:
import sys, subprocess, os, re
def run(cmd):
    subprocess.run(cmd, check=True)

if "COLAB_" not in "".join(os.environ.keys()):
    run([sys.executable, "-m", "pip", "install", "unsloth"])
else:
    import torch
    v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    run([sys.executable, "-m", "pip", "install", "--no-deps", "bitsandbytes", "accelerate", xformers, "peft", "trl", "triton", "cut_cross_entropy", "unsloth_zoo"])
    run([sys.executable, "-m", "pip", "install", "sentencepiece", "protobuf", "datasets>=3.4.1,<4.0.0", "huggingface_hub>=0.34.0", "hf_transfer"])
    run([sys.executable, "-m", "pip", "install", "--no-deps", "unsloth"])
run([sys.executable, "-m", "pip", "install", "transformers==4.56.2"])
run([sys.executable, "-m", "pip", "install", "--no-deps", "trl==0.22.2"])

In [6]:
from google.colab import files
uploaded = files.upload()
import json, os
local_path = list(uploaded.keys())[0]
dataset_path = '/content/' + local_path

Saving mdn_filtered_knowledge_base.json to mdn_filtered_knowledge_base.json


In [8]:
def chunk_text(input, max_len=1500):
    parts = input.split('\n\n')
    chunks = []
    buf = ''
    for p in parts:
        seg = p.strip()
        if not seg:
            continue
        if len(buf) + len(seg) + (2 if buf else 0) <= max_len:
            buf = (buf + ('\n\n' if buf else '') + seg)
            continue
        if buf:
            chunks.append(buf)
            buf = ''
        if len(seg) <= max_len:
            chunks.append(seg)
            continue
        s = seg
        while len(s) > max_len:
            cut = s.rfind('.', 0, max_len)
            idx = cut + 1 if cut > 200 else max_len
            chunks.append(s[:idx].strip())
            s = s[idx:].strip()
        if s:
            chunks.append(s)
    if buf:
        chunks.append(buf)
    return chunks

In [10]:
out_path = '/content/finetune_instructions.jsonl'
items = json.load(open(dataset_path, 'r', encoding='utf-8'))
f = open(out_path, 'w', encoding='utf-8')
for it in items:
    topic = it.get('topic') or ''
    summary = it.get('summary') or ''
    detail = it.get('detailed_knowledge') or ''
    content = ('\n\n').join([x for x in [summary, detail] if x])
    for c in chunk_text(content, 1500):
        rec = { 'instruction': 'Giải thích chi tiết: ' + topic, 'input': '', 'output': c }
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')
f.close()
print(out_path)

/content/finetune_instructions.jsonl


## Unsloth Programmatic QLoRA

In [ ]:
import torch
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset
from transformers import Trainer, TrainingArguments

# Custom Data Collator to handle already padded inputs
class CustomDataCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, examples):
        input_ids = torch.tensor([e['input_ids'] for e in examples], dtype=torch.long)
        attention_mask = torch.tensor([e['attention_mask'] for e in examples], dtype=torch.long)

        labels = input_ids.clone()
        # Set padding tokens to -100 so they are ignored in loss calculation
        if self.tokenizer.pad_token_id is not None:
            labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }


model_id = 'unsloth/gemma-3-4b-it-unsloth-bnb-4bit'
model, tokenizer = FastLanguageModel.from_pretrained(model_name=model_id, max_seq_length=2048, dtype=None, load_in_4bit=True)
tokenizer = get_chat_template(tokenizer, chat_template='gemma-3')
model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    target_modules=['q_proj','k_proj','v_proj','o_proj'],
    lora_alpha=16,
    lora_dropout=0.0,
    bias='none',
    use_gradient_checkpointing=True,
    random_state=3407,
    use_rslora=True,
    loftq_config=None
)
dataset = load_dataset('json', data_files='/content/finetune_instructions.jsonl', split='train')

def to_chat_text(ex):
    inst = ex['instruction']
    inp = ex.get('input','')
    out = ex['output']
    user = inst + (('\n\n' + inp) if inp else '')
    convo = [
      { 'role': 'user', 'content': user },
      { 'role': 'assistant', 'content': out }
    ]
    txt = tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False).removeprefix('<bos>')
    return { 'text': txt }

dataset = dataset.map(to_chat_text, remove_columns=dataset.column_names)
dc = CustomDataCollator(tokenizer) # Use custom data collator
args = TrainingArguments(
    output_dir='/content/outputs/gemma3-4b-mdn-lora',
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=2,
    learning_rate=2e-4,
    logging_steps=50,
    save_steps=200,
    save_total_limit=2,
    bf16=False # Changed to False as the current GPU does not support bf16
)

def tokenize(ex):
    return tokenizer(ex['text'], truncation=True, padding='max_length', max_length=2048)

tokenized = dataset.map(tokenize, batched=True, remove_columns=['text'])
trainer = Trainer(model=model, args=args, train_dataset=tokenized, data_collator=dc)
trainer.train()
trainer.save_model('/content/outputs/gemma3-4b-mdn-lora')
tokenizer.save_pretrained('/content/outputs/gemma3-4b-mdn-lora')

==((====))==  Unsloth 2025.11.3: Fast Gemma3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.
Unsloth: Gemma3 does not support SDPA - switching to fast eager.
Unsloth: Making `base_model.model.model.vision_tower.vision_model` require gradients


Map:   0%|          | 0/24231 [00:00<?, ? examples/s]

Map:   0%|          | 0/24231 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 24,231 | Num Epochs = 2 | Total steps = 3,030
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 47,595,520 of 4,347,674,992 (1.09% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss


## Inference Test

In [ ]:
from transformers import AutoModelForCausalLMfrom peft import PeftModelbase_id = 'unsloth/gemma-3-4b-it-unsloth-bnb-4bit'base = AutoModelForCausalLM.from_pretrained(base_id, load_in_4bit=True, device_map='auto')lora = PeftModel.from_pretrained(base, '/content/outputs/gemma3-4b-mdn-lora')prompt = 'Giải thích cách thêm JavaScript vào HTML'convo = [ { 'role': 'user', 'content': prompt } ]txt = tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=True).removeprefix('<bos>')inputs = tokenizer(txt, return_tensors='pt').to(lora.device)out = lora.generate(**inputs, max_new_tokens=256)print(tokenizer.decode(out[0], skip_special_tokens=True))

## Save to Drive

In [ ]:
from google.colab import drivedrive.mount('/content/drive')import sys, subprocesssubprocess.run(['cp','-r','/content/outputs/gemma3-4b-mdn-lora','/content/drive/MyDrive/gemma3-4b-mdn-lora'], check=True)